# OpenEO algorithms from the ESA APEx Algorithm Catalogue

This notebook renders **real** Earth-observation algorithms from the
[ESA APEx Algorithm Catalogue](https://algorithm-catalogue.apex.esa.int) as live
layers in JupyterGIS.

The catalogue publishes algorithms as openEO **User-Defined Processes** (UDPs) —
self-contained openEO process graphs — in the public
[`ESA-APEx/apex_algorithms`](https://github.com/ESA-APEx/apex_algorithms)
repository. A subset are pure band-math / index algorithms whose openEO process
vocabulary is supported by
[`titiler-openeo`](https://sentinel-hub.github.io/titiler-openeo), so we can serve
their **original** graphs straight into JupyterGIS — only the Sentinel-2
`load_collection` binding changes.

The helper module [`esa_catalogue.py`](./esa_catalogue.py) fetches a UDP graph
verbatim from GitHub, rebinds its `load_collection` to the collection your
titiler-openeo exposes, **remaps the graph's band names to whatever your backend
actually exposes** (catalogue graphs ask for bare Sentinel-2 bands like `B02`,
while many backends expose them suffixed by resolution, e.g. `B02_10m`), resolves
the graph's parameters (to the catalogue author's own defaults unless you
override them), and returns an openEO `DataCube` ready for
`add_openeo_tile_layer`.

## Prerequisites

Same backend as [`99-OpenEO-titiler-local.ipynb`](./99-OpenEO-titiler-local.ipynb):
a running **titiler-openeo** server. It is the only openEO backend that
advertises the `XYZ` secondary tile service JupyterGIS needs to render an openEO
layer (Copernicus Data Space / openEO Platform expose these algorithms only as
batch jobs, not live tiles). Your server must expose a **Sentinel-2 L2A**
collection; set `SERVER_URL` and `COLLECTION` below to match it.

> **Backend support varies.** These are valid catalogue UDPs, but a given
> titiler-openeo build may still reject some of them — its process-graph
> validator, pipeline-item limit, or an unsupported band-math shape. The three
> algorithms showcased below are the ones confirmed to validate **and** render a
> real tile on a reference titiler-openeo; `list_algorithms()` marks every
> confirmed-working graph with a `✓`.

⚠️ This notebook needs a live titiler-openeo backend and cannot run in a
JupyterLite / Notebook.link context unless it points at a remote server.

In [ ]:
SERVER_URL = "http://127.0.0.1:8080/"
USERNAME = "test"
PASSWORD = "test"  # noqa: S105

# Collection id your titiler-openeo exposes for Sentinel-2 L2A. The catalogue
# graphs target the Copernicus id "SENTINEL2_L2A"; a titiler-openeo backed by
# Element-84 Earth Search uses "sentinel-2-l2a". Adjust to match your server.
COLLECTION = "sentinel-2-l2a"

In [ ]:
import importlib

import esa_catalogue
import openeo
from jupytergis import GISDocument

# Reload the local helper so edits to esa_catalogue.py are picked up without a
# full kernel restart (the module may already be cached from an earlier run),
# then rebind the names we use so no import sits after the reload statement.
importlib.reload(esa_catalogue)
build_datacube = esa_catalogue.build_datacube
default_center = esa_catalogue.default_center
list_algorithms = esa_catalogue.list_algorithms

connection = openeo.connect(SERVER_URL)
connection.authenticate_basic(USERNAME, PASSWORD)

## Available algorithms

Catalogue UDPs whose process vocabulary titiler-openeo supports. A `✓` marks the
graphs confirmed to validate **and** render on a reference titiler-openeo — the
safe ones to start with. Unmarked entries are still valid UDPs, but your backend
build may reject them (see the note above). Pass any id below to `build_datacube`.

In [ ]:
list_algorithms()

## ❄️ Snow — Normalized Difference Snow Index (NDSI)

Classifies snow from Sentinel-2 L2A using the green/SWIR Normalized Difference Snow Index; snow pixels are rendered in blue over a brightened true-colour background.

**Catalogue source:** [`ndsi`](https://github.com/ESA-APEx/apex_algorithms/blob/main/algorithm_catalog/developmentseed/ndsi/openeo_udp/ndsi.json) —
we run this graph verbatim (only the Sentinel-2 collection is rebound). The map
is centred on the algorithm author's own default area of interest.

In [ ]:
name = "ndsi"
cube = build_datacube(connection, name, collection=COLLECTION)

latitude, longitude = default_center(name)
doc = GISDocument(latitude=latitude, longitude=longitude, zoom=11)
await doc.ready()
doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Satellite basemap",
)
doc.add_openeo_tile_layer(cube, name="NDSI (snow)")
doc

## ❄️ Snow classification — NDSI + NDVI

Per-pixel snow classification: combines the Normalized Difference Snow Index with
an NDVI vegetation check to separate snow from bright, snow-free surfaces, over a
true-colour background.

**Catalogue source:** [`snow_classifier`](https://github.com/ESA-APEx/apex_algorithms/blob/main/algorithm_catalog/developmentseed/snow_classifier/openeo_udp/snow_classifier.json) —
we run this graph verbatim (only the Sentinel-2 collection is rebound and band
names remapped). The map is centred on the algorithm author's own default area of
interest.

In [ ]:
name = "snow_classifier"
cube = build_datacube(connection, name, collection=COLLECTION)

latitude, longitude = default_center(name)
doc = GISDocument(latitude=latitude, longitude=longitude, zoom=11)
await doc.ready()
doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Satellite basemap",
)
doc.add_openeo_tile_layer(cube, name="Snow classifier")
doc

## 💧 Water — Simple Water Bodies Mapping (SWBM)

Delineates open water from Sentinel-2 L2A using a simple multi-band water index,
highlighting lakes, rivers and reservoirs.

**Catalogue source:** [`swbm`](https://github.com/ESA-APEx/apex_algorithms/blob/main/algorithm_catalog/developmentseed/swbm/openeo_udp/swbm.json) —
we run this graph verbatim (only the Sentinel-2 collection is rebound and band
names remapped). The map is centred on the algorithm author's own default area of
interest.

In [ ]:
name = "swbm"
cube = build_datacube(connection, name, collection=COLLECTION)

latitude, longitude = default_center(name)
doc = GISDocument(latitude=latitude, longitude=longitude, zoom=11)
await doc.ready()
doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Satellite basemap",
)
doc.add_openeo_tile_layer(cube, name="Water bodies (SWBM)")
doc

## Try another algorithm

Swap in any id printed by `list_algorithms()` above. Start with the `✓` ones —
they're confirmed to render on a reference titiler-openeo. Unmarked ids are valid
catalogue UDPs but your backend build may reject them (a process-graph validator
false-positive, a pipeline-item limit, or an unsupported band-math shape); if one
errors, that's a backend limitation, not a broken graph.

You can override the area of interest or date range on any algorithm via the
`spatial_extent=` / `temporal_extent=` arguments — handy for algorithms that ship
without a default AOI, or to point a verified one at your own region:

```python
cube = build_datacube(
    connection,
    "swbm",
    collection=COLLECTION,
    spatial_extent={"west": -120.5, "south": 38.9, "east": -120.2, "north": 39.1},
    temporal_extent=["2021-08-20", "2021-09-10"],
)
```

Band names are remapped to your backend automatically; pass `bands=[...]` to
override, or `remap_bands=False` to disable it.